# C5 — Calibração e Escolha de Limiar

Calibra os finalistas e escolhe o limiar operacional usando **somente treino e validação** (`val_ref`). Com `HONEST_CALIB_SPLIT = True`, o `val_ref` é dividido em duas metades estratificadas e disjuntas: `val_cal` ajusta o calibrador e `val_meas` mede o Brier e escolhe o limiar. O finalista é o congelado em `selecao_modelos.finalista_escolhido` e o limiar segue `limiar.metrica` e `limiar.min_recall` do YAML. Não abre o teste e não escreve no YAML.

**Pré-requisito:** `pip install -e .` com scikit-learn ≥ 1.6.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact, split_partitions
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas, brier
from src.evaluation.calibration import curva_confiabilidade, ece, mce, brier_decomposicao
from src.models.factory import build_finalista
from src.models.calibration import calibrar, escolher_limiar

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

SEED      = int(C.SEED)
G7_STATUS = str(C.cfg("gates", "G7_calibracao", "status") or "pendente")
G8_STATUS = str(C.cfg("gates", "G8_limiar", "status") or "pendente")

print(C.resumo())

ROOT_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
DATA_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data
RAW_DIR           = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrad

## Carregar artefacto e preparar validação

Se `HONEST_CALIB_SPLIT = True`, `val_cal` ajusta o calibrador e
`val_meas` mede — evita o viés otimista de calibrar e medir no mesmo
conjunto. Caso contrário, ambos são a validação inteira.

In [2]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "final").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES].copy()
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

check_leakage(X_train)
check_leakage(X_val)

HONEST_CALIB_SPLIT = True
CALIB_SPLIT_FRAC   = 0.5
CALIB_SPLIT_SEED   = SEED

if HONEST_CALIB_SPLIT:
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - CALIB_SPLIT_FRAC,
        random_state=CALIB_SPLIT_SEED,
        stratify=y_val.to_numpy(),
    )
    X_val_cal  = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal  = y_val.iloc[idx_cal].reset_index(drop=True)
    X_val_meas = X_val.iloc[idx_meas].reset_index(drop=True)
    y_val_meas = y_val.iloc[idx_meas].reset_index(drop=True)
    CALIB_SPLIT_HASH = hashlib.sha256(
        np.asarray(idx_cal, dtype=np.int64).tobytes()
    ).hexdigest()[:16]
else:
    X_val_cal, y_val_cal = X_val, y_val
    X_val_meas, y_val_meas = X_val, y_val
    CALIB_SPLIT_HASH = None

n_teste = int(sum(int(v) for k, v in manifest["partition_counts"].items() if str(k) in {"teste", "test_ref", "test_pandemic"}))

display(pd.DataFrame({
    "bloco":      ["treino", "val_cal", "val_meas"],
    "n":          [len(X_train), len(X_val_cal), len(X_val_meas)],
    "positivos":  [int(y_train.sum()), int(y_val_cal.sum()), int(y_val_meas.sum())],
}))

print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {n_teste:,} (fechado)")
if CALIB_SPLIT_HASH:
    print(f"split_hash  : {CALIB_SPLIT_HASH}")

,bloco,n,positivos
0,treino,556462,46961
1,val_cal,59621,5032
2,val_meas,59622,5032


cohort_kind : final
features    : 31
teste       : 3,035,074 (fechado)
split_hash  : 9b621e331d8420e1


## Calibrar finalistas e escolher limiar

Para cada finalista da `selecao_modelos.finalistas`, treina no
treino, calibra cada método candidato em `val_cal`, mede Brier em
`val_meas` e escolhe o melhor. Depois escolhe o limiar no mesmo
`val_meas`.

Os nomes do YAML são resolvidos em `src.models.factory` (aliases como
`hist_gradient_boosting → boosting`). Este notebook chama
`build_finalista` diretamente, sem duplicar mapeamentos.

In [ ]:
selecao_cfg = C.cfg("selecao_modelos") or {}
FINALISTAS  = list(selecao_cfg.get("finalistas") or [])
if not FINALISTAS:
    raise RuntimeError("selecao_modelos.finalistas está vazio no YAML.")

FINALISTA_ESCOLHIDO = selecao_cfg.get("finalista_escolhido")
if FINALISTA_ESCOLHIDO is None:
    raise RuntimeError(
        "selecao_modelos.finalista_escolhido está vazio no YAML: congele o finalista (G6) antes de rodar o C5."
    )
if FINALISTA_ESCOLHIDO not in FINALISTAS:
    raise RuntimeError(
        f"selecao_modelos.finalista_escolhido={FINALISTA_ESCOLHIDO!r} não está em selecao_modelos.finalistas={FINALISTAS}."
    )
# Hash da configuração dos modelos: o C6 recusa abrir o teste se o YAML dos modelos mudou depois deste C5.
CONFIG_MODELO_HASH = hashlib.sha256(
    json.dumps(
        {"models": C.cfg("models"), "features": sorted(FEATURES), "finalista": FINALISTA_ESCOLHIDO},
        sort_keys=True, default=str,
    ).encode("utf-8")
).hexdigest()[:16]

calibracao_cfg      = C.cfg("calibracao") or {}
CALIBRACAO_ATIVA    = bool(calibracao_cfg.get("ativa", True))
METODOS_CANDIDATOS  = list(calibracao_cfg.get("metodos_candidatos") or ["sigmoid", "isotonic"])

limiar_cfg            = C.cfg("limiar") or {}
LIMIAR_METRICA        = limiar_cfg.get("metrica") or "f1"
LIMIAR_MIN_PRECISION  = limiar_cfg.get("min_precision")
LIMIAR_MIN_RECALL     = limiar_cfg.get("min_recall")

print(f"finalistas          : {FINALISTAS}")
print(f"calibracao ativa    : {CALIBRACAO_ATIVA} ({METODOS_CANDIDATOS})")
print(f"limiar metrica      : {LIMIAR_METRICA} (min_precision={LIMIAR_MIN_PRECISION}, min_recall={LIMIAR_MIN_RECALL})")

resultados = {}

for nome_yaml in FINALISTAS:
    pipe = build_finalista(nome_yaml)   # alias resolvido dentro da fábrica
    pipe.fit(X_train, y_train)
    prob_base = pipe.predict_proba(X_val_meas)[:, 1]
    brier_base = float(brier(y_val_meas, prob_base))

    calibracoes = {}
    if CALIBRACAO_ATIVA:
        for metodo in METODOS_CANDIDATOS:
            try:
                if metodo == "sigmoid":
                    cal = calibrar(pipe, X_val_cal, y_val_cal, method="sigmoid")
                elif metodo == "isotonic":
                    cal = calibrar(pipe, X_val_cal, y_val_cal, method="isotonic")
                else:
                    raise ValueError(f"Método de calibração inválido: {metodo!r}")
                prob = cal.predict_proba(X_val_meas)[:, 1]
                calibracoes[metodo] = {
                    "pipeline":  cal,
                    "prob_meas": prob,
                    "brier":     float(brier(y_val_meas, prob)),
                }
            except Exception as exc:
                print(f"  {nome_yaml}/{metodo}: falhou ({exc!r})")

    if calibracoes:
        metodo = min(calibracoes, key=lambda m: calibracoes[m]["brier"])
        escolha = {"metodo": metodo, **calibracoes[metodo]}
    else:
        escolha = {"metodo": None, "pipeline": pipe,
                   "prob_meas": prob_base, "brier": brier_base}

    resultados[nome_yaml] = {
        "brier_base":    brier_base,
        "prob_base_meas": prob_base,
        "calibracoes":   calibracoes,
        "escolha":       escolha,
    }

    res = escolher_limiar(
        y_val_meas,
        escolha["prob_meas"],
        metrica=LIMIAR_METRICA,
        min_precision=LIMIAR_MIN_PRECISION,
        min_recall=LIMIAR_MIN_RECALL,
    )
    resultados[nome_yaml]["limiar"] = res

tabela = pd.DataFrame([
    {"finalista": n,
     "metodo_calibracao": r["escolha"]["metodo"],
     "brier_base": r["brier_base"],
     "brier_calibrado": r["escolha"]["brier"],
     "ganho_brier": r["brier_base"] - r["escolha"]["brier"],
     "limiar": r["limiar"]["limiar"],
     "metrica": r["limiar"]["metrica"],
     "valor_metrica": r["limiar"]["valor"]}
    for n, r in resultados.items()
])
display(tabela)
tabela_cw = pd.DataFrame()

## Comparação de `class_weight` (item #5)

O `class_weight="balanced"` desloca as probabilidades brutas para cima da prevalência real (8,4% na coorte definitiva). A calibração em `val_cal` corrige — mas ainda não foi medido se **treinar sem peso** entrega Brier/ECE iguais ou melhores com PR-AUC comparável.

Este bloco treina **as duas variantes com os mesmos hiperparâmetros** (os do YAML vencedor) e o **mesmo método de calibração** já escolhido pelo C_5, mede em `val_meas` e mostra o comparativo. Não altera a escolha do C_5 — apenas documenta o efeito e alimenta a decisão do item #5.

In [4]:
if CALIBRACAO_ATIVA:
    comparacao_class_weight = []

    for nome_yaml in FINALISTAS:
        metodo = resultados[nome_yaml]["escolha"]["metodo"]

        if metodo is None:
            print(f"  {nome_yaml}: sem calibrador válido — comparação pulada")
            continue

        for cw in ("balanced", None):
            pipe_cw = build_finalista(nome_yaml, class_weight=cw)
            pipe_cw.fit(X_train, y_train)
            cal_cw = calibrar(pipe_cw, X_val_cal, y_val_cal, method=metodo)
            prob_cw = cal_cw.predict_proba(X_val_meas)[:, 1]
            m = metricas_completas(y_val_meas, prob_cw, limiar=0.5)

            comparacao_class_weight.append({
                "finalista":    nome_yaml,
                "metodo":       metodo,
                "class_weight": cw,
                "brier":        m["brier"],
                "pr_auc":       m["pr_auc"],
                "roc_auc":      m["auc"],
                "ece":          float(ece(y_val_meas, prob_cw, n_bins=10)),
            })

    if comparacao_class_weight:
        tabela_cw = (
            pd.DataFrame(comparacao_class_weight)
            .sort_values(["finalista", "brier"])
            .reset_index(drop=True)
        )
        display(tabela_cw)

        # Um "veredito" por finalista, para o item #5
        veredito_cw = (
            tabela_cw
            .sort_values("brier")
            .groupby("finalista", as_index=False)
            .first()[["finalista", "class_weight", "brier", "pr_auc"]]
        )
        print("\nVeredito (menor Brier pós-calibração):")
        display(veredito_cw)
    else:
        tabela_cw = pd.DataFrame()
        print("Nenhuma comparação produzida — calibração ausente para todos os finalistas.")
else:
    tabela_cw = pd.DataFrame()
    print("Calibração desativada — comparação de class_weight não executada.")

,finalista,metodo,class_weight,brier,pr_auc,roc_auc,ece
0,hist_gradient_boosting,sigmoid,NaN,0.064229,0.341003,0.857778,0.002600
1,hist_gradient_boosting,sigmoid,balanced,0.064338,0.336231,0.858010,0.003922
2,logistic,isotonic,NaN,0.065782,0.298486,0.846557,0.002568
3,logistic,isotonic,balanced,0.065828,0.298184,0.846657,0.002122



Veredito (menor Brier pós-calibração):


,finalista,class_weight,brier,pr_auc
0,hist_gradient_boosting,balanced,0.064229,0.341003
1,logistic,balanced,0.065782,0.298486


## Diagnósticos e persistência

Curva de confiabilidade, decomposição de Brier, ECE e MCE em
`val_meas` com o método escolhido. Persistimos tudo em `outputs/`.

In [5]:
diagnosticos = {}
linhas_diag  = []

for nome, r in resultados.items():
    prob = r["escolha"]["prob_meas"]
    d = {
        "curva":  curva_confiabilidade(y_val_meas, prob, n_bins=10),
        "decomp": brier_decomposicao(y_val_meas, prob, n_bins=10),
        "ece":    float(ece(y_val_meas, prob, n_bins=10)),
        "mce":    float(mce(y_val_meas, prob, n_bins=10)),
    }
    diagnosticos[nome] = d
    linhas_diag.append({
        "finalista":   nome,
        "metodo":      r["escolha"]["metodo"] or "sem_calibracao",
        "ece":         d["ece"],
        "mce":         d["mce"],
        "brier":       d["decomp"]["brier"],
        "reliability": d["decomp"]["reliability"],
        "resolution":  d["decomp"]["resolution"],
    })

display(pd.DataFrame(linhas_diag))

,finalista,metodo,ece,mce,brier,reliability,resolution
0,hist_gradient_boosting,sigmoid,0.003922,0.042249,0.064338,0.000037,0.012371
1,logistic,isotonic,0.002122,0.500000,0.065828,0.000056,0.010996


In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"C5_calibracao_limiar_{manifest['cohort_kind']}_{timestamp}"

caminho_tabela       = TABLES_DIR      / f"{prefixo}_escolhas.csv"
caminho_cw           = TABLES_DIR      / f"{prefixo}_comparacao_class_weight.csv"
caminho_predicoes    = PREDICTIONS_DIR / f"{prefixo}_predicoes_val_meas.parquet"
caminho_manifesto    = TABLES_DIR      / f"{prefixo}_manifesto.json"

tabela.to_csv(caminho_tabela, index=False, encoding="utf-8-sig")
if not tabela_cw.empty:
    tabela_cw.to_csv(caminho_cw, index=False, encoding="utf-8-sig")

artefatos_diag = {}
for nome, d in diagnosticos.items():
    c_curva = TABLES_DIR / f"{prefixo}_{nome}_curva.csv"
    d["curva"].to_csv(c_curva, index=False, encoding="utf-8-sig")
    resultados[nome]["limiar"]["tabela"].to_csv(
        TABLES_DIR / f"{prefixo}_{nome}_limiar.csv", index=False, encoding="utf-8-sig",
    )
    artefatos_diag[nome] = {
        "curva":         str(c_curva),
        "ece":           d["ece"],
        "mce":           d["mce"],
        "brier_decomp":  {k: float(v) for k, v in d["decomp"].items()},
    }

predicoes = []
for nome, r in resultados.items():
    predicoes.append(pd.DataFrame({
        "y_true": y_val_meas.to_numpy(), "y_prob": r["prob_base_meas"],
        "modelo": f"{nome}_base", "cohort_kind": manifest["cohort_kind"],
    }))
    if r["escolha"]["metodo"]:
        predicoes.append(pd.DataFrame({
            "y_true": y_val_meas.to_numpy(), "y_prob": r["escolha"]["prob_meas"],
            "modelo": f"{nome}_{r['escolha']['metodo']}", "cohort_kind": manifest["cohort_kind"],
        }))

pred_val = pd.concat(predicoes, ignore_index=True)
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c5 = {
    "cohort_kind":        manifest["cohort_kind"],
    "dataset_sha256":     manifest["dataset_sha256"],
    "feature_columns":    FEATURES,
    "target_column":      ALVO,
    "partition_column":   PARTICAO,
    "n_treino":           int(len(treino)),
    "n_validacao":        int(len(validacao)),
    "n_val_cal":          int(len(X_val_cal)),
    "n_val_meas":         int(len(X_val_meas)),
    "n_teste_fechado":    n_teste,
    "honest_calib_split": bool(HONEST_CALIB_SPLIT),
    "calib_split_frac":   float(CALIB_SPLIT_FRAC),
    "calib_split_seed":   int(CALIB_SPLIT_SEED),
    "calib_split_hash":   CALIB_SPLIT_HASH,
    "finalistas":         FINALISTAS,
    "calibracao_ativa":   CALIBRACAO_ATIVA,
    "metodos_candidatos": METODOS_CANDIDATOS,
    "finalista_escolhido": FINALISTA_ESCOLHIDO,
    "config_modelo_hash": CONFIG_MODELO_HASH,
    "limiar_min_recall": LIMIAR_MIN_RECALL,
    "escolhas": {
        n: {"metodo": r["escolha"]["metodo"],
            "brier_base":      r["brier_base"],
            "brier_calibrado": r["escolha"]["brier"],
            "limiar":          r["limiar"]["limiar"],
            "limiar_metrica":  r["limiar"]["metrica"]}
        for n, r in resultados.items()
    },
    "comparacao_class_weight": (
        tabela_cw.to_dict(orient="records")
        if not tabela_cw.empty else []
    ),
    "diagnosticos":       artefatos_diag,
    "g7_status":          G7_STATUS,
    "g8_status":          G8_STATUS,
    "teste_aberto":       False,
    "yaml_atualizado":    False,
    "artefatos": {
        "tabela":                    str(caminho_tabela),
        "comparacao_class_weight":   str(caminho_cw) if not tabela_cw.empty else None,
        "predicoes_val_meas":        str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c5, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print(f"Tabela     : {caminho_tabela}")
if not tabela_cw.empty:
    print(f"Class_weight: {caminho_cw}")
print(f"Predições  : {caminho_predicoes}")
print(f"Manifesto  : {caminho_manifesto}")

In [ ]:
melhor_nome      = FINALISTA_ESCOLHIDO
metodo_escolhido = resultados[melhor_nome]["escolha"]["metodo"]
limiar_escolhido = resultados[melhor_nome]["limiar"]

# Recomendação de class_weight baseada na comparação (menor Brier pós-calibração)
cw_recomendado_por_finalista = {}
if not tabela_cw.empty:
    for nome in FINALISTAS:
        sub = tabela_cw[tabela_cw["finalista"] == nome]
        if sub.empty:
            continue
        melhor = sub.sort_values("brier").iloc[0]
        cw_recomendado_por_finalista[nome] = melhor["class_weight"]

linhas_proposta = [
    "# Proposta de atualização do YAML — C5 (não aplicada automaticamente).",
    f"# cohort_kind: {manifest['cohort_kind']}",
    f"# G7/G8 no momento: {G7_STATUS}/{G8_STATUS}",
    f"# finalista congelado no YAML (G6): {melhor_nome}",
]
if cw_recomendado_por_finalista:
    linhas_proposta.append(
        "# class_weight recomendado por finalista (menor Brier pós-calibração):"
    )
    for n, cw in cw_recomendado_por_finalista.items():
        linhas_proposta.append(f"#   {n}: {cw}")
linhas_proposta += [
    "calibracao:",
    f"  metodo: {metodo_escolhido}",
    "limiar:",
    f"  otimo: {limiar_escolhido['limiar']}",
    f"  metrica: {limiar_escolhido['metrica']}",
]
proposta_yaml = "\n".join(linhas_proposta)
print(proposta_yaml)

with Run(
    modelo="C5_calibracao_limiar",
    parametros={
        "cohort_kind":        manifest["cohort_kind"],
        "finalistas":         FINALISTAS,
        "calibracao_ativa":   CALIBRACAO_ATIVA,
        "metodos_candidatos": METODOS_CANDIDATOS,
        "honest_calib_split": HONEST_CALIB_SPLIT,
        "limiar_metrica":     LIMIAR_METRICA,
        "min_precision":      LIMIAR_MIN_PRECISION,
        "min_recall":         LIMIAR_MIN_RECALL,
        "teste_aberto":       False,
        "yaml_atualizado":    False,
        "comparacao_class_weight": (
            tabela_cw.to_dict(orient="records")
            if not tabela_cw.empty else []
        ),
        "cw_recomendado_por_finalista": cw_recomendado_por_finalista,
    },
    dataset={
        "n_treino":        int(len(treino)),
        "n_val_cal":       int(len(X_val_cal)),
        "n_val_meas":      int(len(X_val_meas)),
        "n_teste_fechado": n_teste,
    },
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C5 — calibração e limiar escolhidos na validação. Teste não consultado.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**{
        f"{n}_{sufixo}": float(valor)
        for n, res in resultados.items()
        for sufixo, valor in [
            ("brier_base", res["brier_base"]),
            ("brier_calibrado", res["escolha"]["brier"]),
            ("limiar", res["limiar"]["limiar"]),
            ("auc", res["limiar"]["auc"]),
        ]
    })
    if not tabela_cw.empty:
        for _, row in tabela_cw.iterrows():
            sufixo = f"{row['finalista']}_cw_{row['class_weight']}"
            r.metrica(**{
                f"{sufixo}_brier":   float(row["brier"]),
                f"{sufixo}_pr_auc":  float(row["pr_auc"]),
                f"{sufixo}_ece":     float(row["ece"]),
            })

print("Run registrado em:", r.pasta)